# 05 - Final model, plausibility check, interactive demo and environment


**What this notebook does -** Trains and saves the final CNN used by the demo, runs the plausibility check (does the CNN confuse activities whose pressure patterns look alike?), launches the interactive demo, and writes a pinned `requirements.txt`.

## Setup

In [1]:
import os
if not os.path.exists("/content/project"):
    !git clone https://github.com/BHAGI-99/CS5998-Capstone-Project-P.B.W.Nanayakkara-268472T.git /content/project
%cd /content/project
!git pull
!python src/download_data.py
!python src/segment.py

Cloning into '/content/project'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 192 (delta 60), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 4.00 MiB | 8.71 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/project
Already up to date.
Cloning into 'data/raw/Smart-Sleeve-Dataset'...
remote: Enumerating objects: 445, done.
remote: Counting objects: 100% (445/445), done.
remote: Compressing objects: 100% (318/318), done.
remote: Total 445 (delta 127), reused 443 (delta 127), pack-reused 0 (from 0)
Receiving objects: 100% (445/445), 87.33 MiB | 16.04 MiB/s, done.
Resolving deltas: 100% (127/127), done.
Updating files: 100% (287/287), done.
Done. Data folder: data/raw/Smart-Sleeve-Dataset/Smart-Sleeve Dataset
X (2520, 208, 200) uint16 | classes: 18 | subjects: 14
truncated segments: 123


## Final model
Trains on a fixed 72/8/20 split (seed 42) and saves `models/cnn_final.keras` and the hold-out indices.

In [2]:
!python -m src.train_final

final CNN test accuracy on the fixed hold-out set (n=504): 0.7302
saved models/cnn_final.keras and models/test_indices.npy


## Plausibility check

In [3]:
import os
found = [f for f in ["results/cnn_loso_preds.npy", "results/figures/cnn_loso_preds.npy"] if os.path.exists(f)]
print("LOSO predictions found at:", found if found else "NOT FOUND")
if not found:
    print("-> Run notebook 02 first and upload its results to results/figures/ in the repo, then re-run the setup cell (it pulls the latest repo).")

LOSO predictions found at: ['results/figures/cnn_loso_preds.npy']


In [4]:
!python -m src.plausibility 2>&1 | grep -v -i warn | tee results/plausibility_summary.txt

LOSO accuracy from the saved predictions: 0.6472
Spearman correlation between map similarity and confusion rate over the 153 activity pairs: rho=0.49 (p=1e-10)

Most confused activity pairs (symmetric confusion rate, map cosine similarity):
  12 Hold cheeks                    <-> 13 Play mobile phone               confusion=0.22  similarity=0.89
   9 Lean forward at work           <-> 16 Sit with hands on the armrests  confusion=0.13  similarity=0.86
  16 Sit with hands on the armrests <-> 17 Sit leaning to the right        confusion=0.13  similarity=0.79
   2 Fold arms                      <->  7 Hug a doll                      confusion=0.13  similarity=0.89
  14 Write with a hunchback         <-> 15 Write with a straight back      confusion=0.13  similarity=0.96
  15 Write with a straight back     <-> 16 Sit with hands on the armrests  confusion=0.11  similarity=0.93
   9 Lean forward at work           <-> 14 Write with a hunchback          confusion=0.11  similarity=0.93
  17 Sit l

## Interactive demo
Choose an activity and a hold-out trial: the prediction, the confidence and the sensors that drove the decision. Try several trials, including a wrong one.

In [5]:
from src.demo import launch
launch()

Dropdown(description='Activity:', options=(('1. Stand', 0), ('2. Fold arms', 1), ('3. Think', 2), ('4. Side ag…

IntSlider(value=1, description='Trial:', max=1, min=1)

Output()

## Pin the package versions

In [6]:
import importlib.metadata as md, sys
packages = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "seaborn", "ipywidgets", "tensorflow", "openpyxl"]
lines = []
for p in packages:
    try: lines.append(f"{p}=={md.version(p)}")
    except Exception: lines.append(p)
open("requirements.txt", "w").write("\n".join(lines) + "\n")
print(open("requirements.txt").read(), "Python", sys.version.split()[0])

numpy==2.4.6
pandas==2.2.3
scipy==1.16.3
scikit-learn==1.6.1
matplotlib==3.10.0
seaborn==0.13.2
ipywidgets==7.7.1
tensorflow==2.21.0
openpyxl==3.1.5
 Python 3.13.16


## Self-check and download

In [7]:
import os
from src.demo import explain_trial, _load
ok = {"final model saved": os.path.exists("models/cnn_final.keras"), "hold-out indices saved": os.path.exists("models/test_indices.npy"),
      "plausibility summary saved": os.path.exists("results/plausibility_summary.txt"), "pinned requirements written": os.path.getsize("requirements.txt") > 40}
te = _load()["te"]
for i in te[:3]:
    pred, conf = explain_trial(int(i), show=False)
    ok[f"demo works on trial {int(i)}"] = (0 <= pred < 18) and (0 < conf <= 1)
for k, v in ok.items(): print("PASS" if v else "FAIL", "-", k)
assert all(ok.values()), "A check failed - read the FAIL lines above"
print("\nNotebook 05 self-check: ALL PASS")

PASS - final model saved
PASS - hold-out indices saved
PASS - plausibility summary saved
PASS - pinned requirements written
PASS - demo works on trial 1403
PASS - demo works on trial 1618
PASS - demo works on trial 1088

Notebook 05 self-check: ALL PASS


In [8]:
!zip -j results_05.zip results/plausibility_summary.txt results/figures/plausibility_*.png
from google.colab import files
files.download("results_05.zip")
files.download("models/cnn_final.keras")
files.download("models/test_indices.npy")
files.download("requirements.txt")

  adding: plausibility_summary.txt (deflated 66%)
  adding: plausibility_activity_maps.png (deflated 23%)
  adding: plausibility_similarity_vs_confusion.png (deflated 14%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>